
# Flight Delays — Vanilla ANN (Testing/Inference)

This Colab-ready notebook contains both the training pipeline and the testing/inference pipeline for a multi-output ANN that predicts flight delay minutes by cause: Carrier, Weather, Airport, Security, Late Aircraft.


In [ ]:

# 0) Mount Google Drive (required in Colab)
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

# 1) Imports & Reproducibility seed
import os, json
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow import keras

BASE = "/content/drive/MyDrive/DeepLearning_Project"
DATA_DIR   = f"{BASE}/data"
MODELS_DIR = f"{BASE}/models"
OUT_DIR    = f"{BASE}/outputs"
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:

# 2) Loading (metadata, scaler, model)
with open(f"{MODELS_DIR}/metadata.json") as f:
    meta = json.load(f)
feature_cols = meta["feature_cols"]
target_cols  = meta["target_cols"]
keys         = meta["keys"]
global_mean  = meta["global_mean"]

scaler = joblib.load(f"{MODELS_DIR}/standard_scaler.pkl")
model  = keras.models.load_model(f"{MODELS_DIR}/best_val_model.keras")

# Lookups
paths = {
    "airline_month": f"{MODELS_DIR}/feats_airline_month.csv",
    "origin_slot":   f"{MODELS_DIR}/feats_origin_slot.csv",
    "route_dow":     f"{MODELS_DIR}/feats_route_dow.csv"
}
lookups = {k: pd.read_csv(p) for k,p in paths.items() if os.path.exists(p)}
list(lookups.keys())


[]

In [ ]:

# 3) Load CSV
csv_path = f"{DATA_DIR}/DelayFlights-cleaned-handpick.csv"  # o el archivo nuevo que quieras puntuar
if not os.path.exists(csv_path):

    for root, _, files in os.walk(DATA_DIR):
        for f in files:
            if f.lower().endswith('.csv'):
                csv_path = os.path.join(root, f)
                break
print('CSV test:', csv_path)

df = pd.read_csv(csv_path).reset_index(drop=True)

# Targets
has_targets = all([c in df.columns for c in target_cols])

# route
if keys.get("origin_city") and keys.get("destination"):
    df["route"] = df[keys["origin_city"]].astype(str) + "_" + df[keys["destination"]].astype(str)
else:
    df["route"] = "UNK"

# lookups
Xdf = df.copy()
if "airline_month" in lookups and keys.get("airline_code") and keys.get("month"):
    Xdf = Xdf.merge(lookups["airline_month"], on=[keys["airline_code"], keys["month"]], how='left')
if "origin_slot" in lookups and keys.get("origin_city") and keys.get("depart_time_block"):
    Xdf = Xdf.merge(lookups["origin_slot"], on=[keys["origin_city"], keys["depart_time_block"]], how='left')
if "route_dow" in lookups and keys.get("day_of_week"):
    Xdf = Xdf.merge(lookups["route_dow"], on=["route", keys["day_of_week"]], how='left')

for c in ["feat_airline_month_mean","feat_origin_slot_mean","feat_route_dow_mean"]:
    if c in Xdf.columns:
        Xdf[c] = Xdf[c].fillna(global_mean)

missing = [c for c in feature_cols if c not in Xdf.columns]
assert not missing, f"Missing feature columns in test data": {missing}"
X = Xdf[feature_cols].values
X_sc = scaler.transform(X)


CSV test: /content/drive/MyDrive/DeepLearning_Project/data/DelayFlights-cleaned.csv


In [ ]:
# 4) Prediction
pred = model.predict(X_sc)
pred_cols = [f"pred_{t}" for t in target_cols]
pred_df = pd.DataFrame(pred, columns=pred_cols)

259975/259975 ━━━━━━━━━━━━━━━━━━━━ 537s 2ms/step


In [ ]:
out_csv = f"{OUT_DIR}/predictions_test.csv"
pred_df.to_csv(out_csv, index=False)
print('Predictions saved in:', out_csv)

if has_targets:
    y_true = df[target_cols].values
    mae_g  = mean_absolute_error(y_true, pred)
    rmse_g = np.sqrt(mean_squared_error(y_true, pred))
    print(f"GLOBAL — MAE: {mae_g:.3f} | RMSE: {rmse_g:.3f}")
    for i, col in enumerate(target_cols):
        mae_i  = mean_absolute_error(y_true[:, i], pred[:, i])
        rmse_i = np.sqrt(mean_squared_error(y_true[:, i], pred[:, i]))
        print(f"{col:20s} — MAE: {mae_i:.3f} | RMSE: {rmse_i:.3f}")
else:
    print('No targets present in this CSV; generated predictions only.')

Predicciones guardadas en: /content/drive/MyDrive/DeepLearning_Project/outputs/predictions_test.csv
GLOBAL — MAE: 6.795 | RMSE: 34.711
Carrier Delay        — MAE: 12.094 | RMSE: 53.508
Weather Delay        — MAE: 2.235 | RMSE: 22.905
Airport Delay        — MAE: 7.008 | RMSE: 23.727
Security Delay       — MAE: 0.284 | RMSE: 2.298
Late Aircraft Delay  — MAE: 13.895 | RMSE: 45.801
